# 00 — Collecte des données historiques DE40 5min

**Objectif** : récupérer le maximum de données historiques disponibles pour l'epic `DE40` (DAX 40) à la résolution `MINUTE_5` via l'API Capital.com demo.

**Stratégie de pagination** :
- L'API limite le nombre de candles par appel
- On pagine **jour par jour**, fenêtres de 45 min couvrant la session DAX
- Session DAX : 09:00–17:30 CET → on collecte **07:00–16:35 UTC** pour couvrir heure d'été et d'hiver

**Output** : `data/dax_live_5min.csv`

## 1. Imports

In [ ]:
import sys
import os
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from datetime import datetime, timedelta, timezone

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

from src.service.collector.session_manager import SessionManager
from src.service.collector.api_client import CapitalClient

print(f"Imports OK — ROOT = {ROOT}")

## 2. Chargement des credentials (.env)

Le fichier `.env` doit contenir :
```
CAPITAL_API_KEY=...
CAPITAL_IDENTIFIER=...
CAPITAL_PASSWORD=...
```

In [ ]:
load_dotenv(ROOT / ".env")

API_KEY    = os.getenv("CAPITAL_API_KEY")
IDENTIFIER = os.getenv("CAPITAL_IDENTIFIER")
PASSWORD   = os.getenv("CAPITAL_PASSWORD")

assert API_KEY and IDENTIFIER and PASSWORD, (
    "Variables manquantes dans .env : CAPITAL_API_KEY, CAPITAL_IDENTIFIER, CAPITAL_PASSWORD"
)
print("Credentials chargés")

## 3. Connexion à l'API

In [ ]:
session = SessionManager(API_KEY, IDENTIFIER, PASSWORD, ping=True)  # ping=True garde la session active
client  = CapitalClient(session)
print("Client prêt")

## 4. Paramètres de collecte

In [ ]:
EPIC       = "DE40"
RESOLUTION = "MINUTE_5"
WINDOW_MIN = 45

# 07:00–16:35 UTC couvre la session DAX en heure d'été et d'hiver
SESSION_START = "07:00"
SESSION_END   = "16:35"

SLEEP_BETWEEN_CALLS = 0.3

end_date   = datetime.now(timezone.utc).date()
start_date = end_date - timedelta(days=380)

business_days = pd.bdate_range(start=start_date, end=end_date).date.tolist()
business_days = list(reversed(business_days))

print(f"Business days à collecter : {len(business_days)}")
print(f"Période : {business_days[-1]} → {business_days[0]}")

## 5. Fonction de parsing des candles

In [ ]:
def parse_candles(prices: list) -> pd.DataFrame:
    """Convertit la liste de candles API en DataFrame. Utilise mid = (bid+ask)/2."""
    rows = []
    for p in prices:
        ts = pd.Timestamp(p["snapshotTimeUTC"], tz="UTC")
        rows.append({
            "datetime":  ts,
            "open":  (p["openPrice"]["bid"]  + p["openPrice"]["ask"])  / 2,
            "high":  (p["highPrice"]["bid"]  + p["highPrice"]["ask"])  / 2,
            "low":   (p["lowPrice"]["bid"]   + p["lowPrice"]["ask"])   / 2,
            "close": (p["closePrice"]["bid"] + p["closePrice"]["ask"]) / 2,
            "volume": p.get("lastTradedVolume", 0),
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df.set_index("datetime", inplace=True)
        df.sort_index(inplace=True)
    return df

print("Fonction parse_candles définie")

## 6. Collecte paginée

In [ ]:
all_frames  = []
total_calls = 0
total_bars  = 0

for day in business_days:
    day_str = str(day)
    session_start = datetime.strptime(f"{day_str}T{SESSION_START}:00", "%Y-%m-%dT%H:%M:%S")
    session_end   = datetime.strptime(f"{day_str}T{SESSION_END}:00",   "%Y-%m-%dT%H:%M:%S")

    day_frames  = []
    window_from = session_start

    while window_from < session_end:
        window_to = min(window_from + timedelta(minutes=WINDOW_MIN), session_end)
        from_str  = window_from.strftime("%Y-%m-%dT%H:%M:%S")
        to_str    = window_to.strftime("%Y-%m-%dT%H:%M:%S")

        try:
            result = client.get_candles_range(
                epic=EPIC, from_dt=from_str, to_dt=to_str, resolution=RESOLUTION
            )
            prices = result.get("prices", [])
        except ValueError as e:
            err_str = str(e)
            if "404" in err_str or "daterange" in err_str:
                prices = []
            else:
                print(f"Erreur {day_str} {from_str}: {e}")
                prices = []

        total_calls += 1
        if prices:
            day_frames.append(parse_candles(prices))
            total_bars += len(prices)

        window_from = window_to
        time.sleep(SLEEP_BETWEEN_CALLS)

    if day_frames:
        all_frames.append(pd.concat(day_frames))
        print(f"  {day_str} : {sum(len(f) for f in day_frames)} bars")
    else:
        print(f"  {day_str} : fermé / aucune donnée")

print(f"\nTotal appels : {total_calls}")
print(f"Total bars   : {total_bars}")

## 7. Assemblage et sauvegarde

In [ ]:
if not all_frames:
    raise RuntimeError("Aucune donnée collectée — vérifier les credentials et l'epic")

df = pd.concat(all_frames)
df = df[~df.index.duplicated(keep="first")]
df.sort_index(inplace=True)

# Convertir en heure de Berlin (CET/CEST)
df.index = df.index.tz_convert("Europe/Berlin")

print(f"Total candles : {len(df):,}")
print(f"Période       : {df.index.min()} → {df.index.max()}")

output_path = ROOT / "data" / "dax_live_5min.csv"
df.to_csv(output_path)

print(f"\nSauvegardé : {output_path}")
print(f"Taille     : {output_path.stat().st_size / 1024 / 1024:.2f} MB")
df.head(10)

## Résumé

| Item | Valeur |
|------|--------|
| Epic | DE40 (DAX 40) |
| Résolution | MINUTE_5 |
| Timezone | Europe/Berlin (CET/CEST) |
| Output | `data/dax_live_5min.csv` |

**Prochaine étape** → `00b_asrs_live.ipynb`